# LFA1: supplier master, general data (EDA)

One row per supplier. Adds supplier name and address (country, region/state, city) for the geography
question (Q2).

Raw export: 25,055 rows x 197 columns; `sap_data.load` keeps 8.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # src/, for sap_data

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sap_data import load, load_raw, PURCH_ORG, COMPANY_CODE, PERIOD_START, PERIOD_END

pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid")

In [ ]:
lfa1 = load("lfa1")
ekko = load("ekko")
print(lfa1.shape)
display(lfa1.head())
lfa1.dtypes

## 1. Grain and key format

In [ ]:
assert lfa1["Supplier"].is_unique
lfa1["Supplier"].str.len().value_counts()

> **Note.** One row per supplier. Keys are strings of 6–7 digits. Both EKKO and LFA1 keys are cast to
> stripped strings in `sap_data`, so the join can't fail on int-vs-string type differences.

## 2. Missing values

In [ ]:
lfa1.isna().agg(["sum", "mean"]).T.sort_values("sum", ascending=False)

## 3. What's in the master: template copies vs. real entries

In [ ]:
display(lfa1.groupby("Account group")["Created On"].agg(["size", "min", "max"]))
lfa1["Name 1"].value_counts().head(26)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
lfa1["Created On"].dt.to_period("Q").value_counts().sort_index().plot.bar(ax=ax, logy=True)
ax.set(title="Suppliers created per quarter (log scale)", xlabel="", ylabel="Suppliers");

> **Note.** 24,000 of 25,055 records are **24 GBI template suppliers × 1,000 copies** (one per student
> number), created in Aug 2021 under account group `LIEF`. The ~1,050 `KRED` suppliers were created by
> students from Sep 2024 onward. Whole-master statistics (e.g. "51% of suppliers are US") describe the
> training client, not Global Bike's supply base. **Decision:** describe geography only for suppliers
> that actually appear on in-scope POs.

## 4. Geography: whole master

In [ ]:
display(lfa1["Country/Region Key"].value_counts().head(15))
us = lfa1[lfa1["Country/Region Key"] == "US"]
print("US suppliers without Region:", f'{us["Region"].isna().mean():.1%}')
us["Region"].value_counts(dropna=False).head(15)

## 5. Geography: suppliers used on purchasing documents

In [ ]:
used_ids = ekko.loc[ekko["Purch. Doc. Category"] == "F", "Supplier"].dropna().unique()
used = lfa1[lfa1["Supplier"].isin(used_ids)]
print(f"PO suppliers: {len(used_ids)}, found in LFA1: {len(used)}")
print("Missing from LFA1:", sorted(set(used_ids) - set(lfa1["Supplier"])))
used["Country/Region Key"].value_counts()

In [ ]:
used_us = used[used["Country/Region Key"] == "US"]
print("US PO suppliers without Region:", f'{used_us["Region"].isna().mean():.1%}')
display(used_us["Region"].fillna("Unknown").value_counts())
used_us["City"].value_counts().head(20)

In [ ]:
# Country says US, address says Hungary
used_us[used_us["City"].str.contains("Budapest|Debrecen|Szeged|Győr", case=False, na=False)][
    ["Supplier", "Name 1", "City", "Postal Code", "Region"]]

> **Note.** On suppliers that are actually used:
> - Mostly US, then **HU** and DE. The Hungarian share reflects the university's students, not a real
>   overseas sourcing strategy.
> - A noticeable share of US suppliers has **no Region (state)**. "Unknown state" must be its own bar
>   in Q2.
> - City is free text: `Dallas` / `dallas` / `DALLAS` / `Dalas`, and even `Texas` as a city. Some
>   suppliers flagged country US have Budapest addresses. **Decision:** use `Country/Region Key` and
>   `Region` as recorded, don't geocode the city, and list the inconsistencies as a data-quality
>   limitation.
> - Supplier names include exercise placeholders (`Vendor`, `szállító…`, `Bike supplier 000`,
>   `Villám`).

## Summary for the report (LFA1)
- Grain: 1 row per supplier; join from EKKO on `Supplier` (many-to-one). 2 PO suppliers are missing
  from the master.
- The master is dominated by 24,000 template copies, so geography is analysed for **in-scope
  suppliers only**.
- Country = supplier master **address**. It isn't the ship-from location and doesn't measure supply
  risk.
- Region is often blank for US suppliers and city spelling is inconsistent, so report "Unknown state"
  explicitly.

## Appendix: columns left out (slow, ~16 MB file)

In [ ]:
raw = load_raw("lfa1")
informative = [c for c in raw.columns if raw[c].nunique(dropna=True) > 1]
print(f"{raw.shape[1]} columns, {len(informative)} with more than one distinct value:")
print(informative)